Loading the libraries

In [ ]:
import sys
import os
import pandas as pd
import pytest

# Add scripts directory to path
project_root = os.path.abspath('..')  # Go up to project root
scripts_path = os.path.join(project_root, 'scripts')
sys.path.insert(0, scripts_path)

# Now import
from utils import load_data

# Load data
df = load_data()

ModuleNotFoundError: No module named 'ipytest'

In [10]:
df.head()


,id,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,...,hour,minute,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,day_of_year_sin,day_of_year_cos
0,1969564,HH162926,2002-01-01,063XX N WASHTENAW AV,0820,THEFT,$500 AND UNDER,STREET,False,False,...,0,0,0.0,1.0,0.781831,0.62349,0.5,0.866025,0.017213,0.999852
1,1921971,HH105505,2002-01-01,085XX S CRANDON AV,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,False,False,...,0,0,0.0,1.0,0.781831,0.62349,0.5,0.866025,0.017213,0.999852
2,1989110,HH188541,2002-01-01,047XX N HERMITAGE AV,0820,THEFT,$500 AND UNDER,RESIDENCE,False,False,...,0,0,0.0,1.0,0.781831,0.62349,0.5,0.866025,0.017213,0.999852
3,1917950,HH101092,2002-01-01,028XX N PARKSIDE AV,0910,MOTOR VEHICLE THEFT,AUTOMOBILE,STREET,False,False,...,0,0,0.0,1.0,0.781831,0.62349,0.5,0.866025,0.017213,0.999852
4,1950239,HH139701,2002-01-01,066XX S KEDVALE AV,0842,THEFT,AGG: FINANCIAL ID THEFT,RESIDENCE,False,False,...,0,0,0.0,1.0,0.781831,0.62349,0.5,0.866025,0.017213,0.999852


In [ ]:
def test_cyclical_encoding_hour_zero(cleaned_df):
    assert cleaned_df[cleaned_df['hour'] == 0]['hour_sin'].iloc[0] == pytest.approx(0.0, abs=1e-9)
    assert cleaned_df[cleaned_df['hour'] == 0]['hour_cos'].iloc[0] == pytest.approx(1.0, abs=1e-9)


In [19]:
#testing haversine with two identical points (should be 0)
lat1, lon1 = 52.5200, 13.4050  # Berlin
lat2, lon2 = 41.8781, -87.6298  # Chicago
distance = haversine(lat1, lon1, lat2, lon2)
print(f"Haversine distance between identical points: {distance:.2f} km")

Haversine distance between identical points: 7083.44 km


In [4]:
#testing cyclical_distance when values are close to the wrap-around point (e.g., hours in a day)
value1 = 23
value2 = 23
max_value = 24  # For example, hours in a day
cyc_dist = cyclical_distance(value1, value2, max_value)
print(f"Cyclical distance between wrap-around values: {cyc_dist:.4f}")




Cyclical distance between wrap-around values: 0.0000


In [30]:
#testing cyclical_distance when values are close to the wrap-around point (e.g., hours in a day)
value1_1 = 4
value2_2 = 1
max_value = 7  # For example, hours in a day
cyc_dist = cyclical_distance(value1_1, value2_2, max_value)
print(f"Cyclical distance between wrap-around values: {cyc_dist:.4f}")

Cyclical distance between wrap-around values: 0.9749


In [5]:
#sanity check for temporal distance being close

month1, day1, hour1, dow1 = 12, 31, 23, 6  # Dec 31, 11 PM, Saturday
month2, day2, hour2, dow2 = 1, 1, 0, 0       # Jan 1, 12 AM, Sunday

temp_distance = temporal_distance(month1, day1, hour1, dow1, month2, day2, hour2, dow2)
print(f"Temporal distance between Dec 31, 11 PM and Jan 1, 12 AM: {temp_distance:.4f}")

Temporal distance between Dec 31, 11 PM and Jan 1, 12 AM: 0.2311


In [8]:
#sanity check for temporal distance being almost equal

month1_1, day1_1, hour1_1, dow1_1 = 12, 31, 23, 6  # Dec 31, 11 PM, Saturday
month2_1, day2_1, hour2_1, dow2_1 = 12, 31, 23, 6       # Dec 31, 11 PM, Saturday

temp_distance_2 = temporal_distance(month1_1, day1_1, hour1_1, dow1_1, month2_1, day2_1, hour2_1, dow2_1)
print(f"Temporal distance between Dec 31, 11 PM, Saturday and Dec 31, 11 PM, Saturday: {temp_distance_2:.4f}")

Temporal distance between Dec 31, 11 PM, Saturday and Dec 31, 11 PM, Saturday: 0.0000


In [9]:
#sanity check for temporal distance being far apart

month1_2, day1_2, hour1_2, dow1_2 = 6, 31, 12, 4  # Jun 31, 12 PM, Thursday
month2_2, day2_2, hour2_2, dow2_2 = 1, 1, 0, 0       # Jan 1, 12 AM, Sunday

temp_distance_3 = temporal_distance(month1_2, day1_2, hour1_2, dow1_2, month2_2, day2_2, hour2_2, dow2_2)
print(f"Temporal distance between Jun 31, 12 PM and Jan 1, 12 AM: {temp_distance_3:.4f}")

Temporal distance between Jun 31, 12 PM and Jan 1, 12 AM: 0.7605


In [15]:
#testing combined distance 
crime1 = {
    'latitude': 52.5200,
    'longitude': 13.4050,
    'month': 7,
    'hour': 12,
    'day_of_week': 0
}

crime2 = {
    'latitude': 41.8,
    'longitude': -87.6,
    'month': 1,
    'hour': 0,
    'day_of_week': 4
}

max_distance = get_spatial_bounds(df)['max_distance']

comb_distance = combined_distance(crime1, crime2, max_distance, alpha=1, beta=0)  # Only spatial

print(f"Combined distance between identical crimes: {comb_distance:.4f}")

haversine_distance = (haversine(crime1['latitude'], crime1['longitude'], crime2['latitude'], crime2['longitude']))/max_distance
print(f"Haversine distance between far away crimes: {haversine_distance:.4f}")

Combined distance between identical crimes: 130.4393
Haversine distance between far away crimes: 130.4393


In [2]:

bounds = get_spatial_bounds(df)

max_distance_km = bounds['max_distance']